In [149]:
# from pathlib import Path
# import sys

# project_root = Path.cwd().parent  # notebooks -> project root
# sys.path.insert(0, str(project_root))

# from clinical_synopsis.embedder import Embedder
# print("embedder import OK")

# to avoid clinical_synopsis.embedder
from pathlib import Path
import sys

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "clinical_synopsis"))

from embedder import Embedder
print("embedder import OK")

embedder import OK


In [150]:
# import rag as rag

# # Look at the first chunk we have in the vector index, take the patient_id attached to that chunk:
# pid = rag.vector_documents[0]["patient_id"]

# result = rag.rag(
#     query="What oncology-related events are documented?",
#     patient_id=pid,
#     search_type="hybrid",
#     num_results=5,
# )

# len(result["search_results"]), result["search_results"][:2]

In [151]:
# print("Answer cost (USD):", result["answer_total_cost_usd"])
# print("Eval cost (USD):  ", result["eval_total_cost_usd"])
# print("Overall cost (USD):", result["overall_total_cost_usd"])

# print("Answer tokens (in/out/total):",
#       result["prompt_tokens"],
#       result["completion_tokens"],
#       result["total_tokens"])

# print("Eval tokens (in/out/total):",
#       result["eval_prompt_tokens"],
#       result["eval_completion_tokens"],
#       result["eval_total_tokens"])

In [152]:
# for i, doc in enumerate(result["search_results"], start=1):
#     print("=" * 80)
#     print("Rank:", i)
#     print("Chunk ID:", doc.get("chunk_id"))
#     print("Patient ID:", doc.get("patient_id"))
#     print("Doc type:", doc.get("doc_type"))
#     print("Title:", doc.get("title"))
#     print("Heading:", doc.get("heading"))
#     print("Text:", doc.get("chunk_text", "")[:500])

In [153]:
# available_patient_ids = sorted({doc["patient_id"] for doc in rag.vector_documents})
# len(available_patient_ids), available_patient_ids[:10]

# 9 patients for ground truth

We pick 9 patients with 3 from each complexity bucket (low, medium, high) for the ground-truth set, in order to cover simple, medium, and complex EHRs.

`n_resources` is the total number of FHIR resources in that patient’s bundle — i.e., how many individual clinical records (Patient, Encounter, Observation, Condition, Procedure, MedicationRequest, DiagnosticReport, etc.) are contained in the JSON file for that patient.

So we see that the higher `n_resources`, the higher `complexity_score`

Remember:
### Note on COMPLEXITY SCORES for each patient

A higher complexity score should reflect more encounters, conditions, procedures, meds, reports for a given patient, as well as a longer follow-up period (e.g., Febrile neutropenia condition gives a clear date (onsetDateTime, recordedDate) and is tied to an encounter, which contributes to follow-up and complexity).

In `sample_mcode_patients.py` a complexity score is computed as:
```python
complexity_score = (
    counts["Encounter"] * 3
    + counts["Observation"] * 1
    + counts["Condition"] * 2
    + counts["Procedure"] * 2
    + counts["MedicationRequest"] * 2
    + counts["MedicationAdministration"] * 2
    + counts["DiagnosticReport"] * 2
    + min(followup_days // 180, 20)
)
```
Which means that:
- Each resource type contributes with a **weight**:
  - Encounters: \(3 \times\) number of encounters (heavier weight).
  - Observations: \(1 \times\) number of observations.
  - Conditions, Procedures, MedicationRequest, MedicationAdministration, DiagnosticReport: each \(2 \times\) their counts.
- Plus a **time component**:
  - `followup_days` is the difference between the first and last clinical dates found in the bundle.
  - `followup_days // 180` converts follow-up into “half-year blocks”.
  - This term is capped at 20, so very long records don’t dominate.



In [154]:
import pandas as pd

manifest_path = project_root / "data" / "processed" / "mcode_breast_sample_50_manifest.csv"

# Load manifest
df = pd.read_csv(manifest_path)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (50, 17)

Columns:
['filename', 'patient_id', 'patient_name', 'n_resources', 'n_encounters', 'n_observations', 'n_conditions', 'n_procedures', 'n_medication_requests', 'n_medication_administrations', 'n_diagnostic_reports', 'first_date', 'last_date', 'followup_days', 'complexity_score', 'complexity_bucket', 'sample_seed']


In [155]:
# Check bucket distribution
print("Bucket counts in full manifest:")
print(df["complexity_bucket"].value_counts())

# Sample 3 patients from each bucket (low, medium, high)
bucket_targets = {"low": 3, "medium": 3, "high": 3}
selected_rows = []

for bucket, n in bucket_targets.items():
    bucket_df = df[df["complexity_bucket"] == bucket].copy()
    if len(bucket_df) < n:
        raise ValueError(f"Not enough patients in bucket '{bucket}' to sample {n}.")

    # Random sample with a fixed seed for reproducibility
    sampled_bucket = bucket_df.sample(n=n, random_state=42)
    selected_rows.append(sampled_bucket)

selected_df = pd.concat(selected_rows).reset_index(drop=True)

print("\nSelected 9 patients (3 per bucket):")
display(selected_df[["patient_id", "patient_name", "complexity_bucket",
                     "n_resources", "complexity_score"]])

# Just the list of patient_ids for later use
selected_patient_ids = selected_df["patient_id"].tolist()
print("\nSelected patient_ids:", selected_patient_ids)

Bucket counts in full manifest:
complexity_bucket
low       17
high      17
medium    16
Name: count, dtype: int64

Selected 9 patients (3 per bucket):


,patient_id,patient_name,complexity_bucket,n_resources,complexity_score
0,d65197b3-056a-2136-b584-77f43c29da3f,Corrie32 Boyle917,low,230,317
1,f3739580-797d-ae04-eebf-aeddb2fc2f64,Florine959 Stark857,low,261,330
2,4736727e-63f4-071a-1516-a49310f5a052,Mónica985 Serrato62,low,436,602
3,29f6beee-162f-0113-7884-72245814693f,Eula461 Crooks415,medium,1854,2786
4,41681ed6-efc5-94c0-1bc0-f60b34dbd31b,Beth967 Cremin516,medium,1843,2800
5,aee216e6-cbe8-eaf2-3241-4bd1e8a01494,Deeann517 Torp761,medium,2191,3340
6,ecc4a7d0-8838-36b4-44ba-676d5a1f7927,Francina926 Von197,high,2945,4458
7,3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678,Rosetta750 Stroman228,high,3055,4536
8,f203e11d-5573-1624-69b8-af8436987b3e,Shawana711 Lakin515,high,3365,4812



Selected patient_ids: ['d65197b3-056a-2136-b584-77f43c29da3f', 'f3739580-797d-ae04-eebf-aeddb2fc2f64', '4736727e-63f4-071a-1516-a49310f5a052', '29f6beee-162f-0113-7884-72245814693f', '41681ed6-efc5-94c0-1bc0-f60b34dbd31b', 'aee216e6-cbe8-eaf2-3241-4bd1e8a01494', 'ecc4a7d0-8838-36b4-44ba-676d5a1f7927', '3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678', 'f203e11d-5573-1624-69b8-af8436987b3e']


In [156]:
selected_patient_ids

['d65197b3-056a-2136-b584-77f43c29da3f',
 'f3739580-797d-ae04-eebf-aeddb2fc2f64',
 '4736727e-63f4-071a-1516-a49310f5a052',
 '29f6beee-162f-0113-7884-72245814693f',
 '41681ed6-efc5-94c0-1bc0-f60b34dbd31b',
 'aee216e6-cbe8-eaf2-3241-4bd1e8a01494',
 'ecc4a7d0-8838-36b4-44ba-676d5a1f7927',
 '3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678',
 'f203e11d-5573-1624-69b8-af8436987b3e']

In [157]:
# look at all oncology chunks in chunks_df:

import sqlite3
import pandas as pd
from pathlib import Path

db_path = Path("../data/retrieval/metadata.db")
patient_ids = selected_patient_ids

conn = sqlite3.connect(db_path)

# Multiple patient_ids: build an IN (...) placeholder list.
if not patient_ids:
    chunks_df = pd.DataFrame()  # avoid invalid SQL: IN ()
else:
    placeholders = ",".join(["?"] * len(patient_ids))
    query = f"""
    SELECT
        chunks.patient_id,
        documents.doc_type,
        documents.title,
        chunks.heading,
        chunks.chunk_id,
        chunks.is_oncology,
        chunks.chunk_text
    FROM chunks
    JOIN documents ON chunks.document_id = documents.document_id
    WHERE chunks.patient_id IN ({placeholders})
    """
    chunks_df = pd.read_sql_query(query, conn, params=patient_ids)

conn.close()

display(chunks_df.head())
print("Number of rows:", len(chunks_df))

onc_chunks = chunks_df[chunks_df["is_oncology"] == 1]
display(onc_chunks.head())
print("Number of oncology chunks:", len(onc_chunks))

,patient_id,doc_type,title,heading,chunk_id,is_oncology,chunk_text
0,29f6beee-162f-0113-7884-72245814693f,conditions,conditions.csv,conditions,82fbc5fb0df955ed9b5861c956dd9d5cdd0a1aa8,1,patient_id: 29f6beee-162f-0113-7884-7224581469...
1,29f6beee-162f-0113-7884-72245814693f,conditions,conditions.csv,conditions,cd9bf67f542dee2c5c6eb4e889086745d239ff7b,0,patient_id: 29f6beee-162f-0113-7884-7224581469...
2,29f6beee-162f-0113-7884-72245814693f,conditions,conditions.csv,conditions,c6f4af530450ec4d38f7f478693b4d62c2b3467c,0,patient_id: 29f6beee-162f-0113-7884-7224581469...
3,29f6beee-162f-0113-7884-72245814693f,conditions,conditions.csv,conditions,9d5bdbca525ce28a75b1ad7deff73853cc1b20eb,0,patient_id: 29f6beee-162f-0113-7884-7224581469...
4,29f6beee-162f-0113-7884-72245814693f,conditions,conditions.csv,conditions,250fb82c086a015ec8fe85d5feeb46806e632e86,0,patient_id: 29f6beee-162f-0113-7884-7224581469...


Number of rows: 14390


,patient_id,doc_type,title,heading,chunk_id,is_oncology,chunk_text
0,29f6beee-162f-0113-7884-72245814693f,conditions,conditions.csv,conditions,82fbc5fb0df955ed9b5861c956dd9d5cdd0a1aa8,1,patient_id: 29f6beee-162f-0113-7884-7224581469...
21,29f6beee-162f-0113-7884-72245814693f,conditions,conditions.csv,conditions,34e083fbe248b34ab7ff75e989fc91da40a6b511,1,patient_id: 29f6beee-162f-0113-7884-7224581469...
931,29f6beee-162f-0113-7884-72245814693f,observations,observations.csv,observations,6c44d78bfe33c70133759592f49f88f9cd26734c,1,patient_id: 29f6beee-162f-0113-7884-7224581469...
932,29f6beee-162f-0113-7884-72245814693f,observations,observations.csv,observations,4f181909c2fad14ebbdc20f28f1af14d5dfb89f9,1,patient_id: 29f6beee-162f-0113-7884-7224581469...
933,29f6beee-162f-0113-7884-72245814693f,observations,observations.csv,observations,b02dbcb96bafeb63dc73b3c40a9f184cb0e781bf,1,patient_id: 29f6beee-162f-0113-7884-7224581469...


Number of oncology chunks: 890


In [158]:
chunks_df.columns

Index(['patient_id', 'doc_type', 'title', 'heading', 'chunk_id', 'is_oncology',
       'chunk_text'],
      dtype='str')

## QUESTION ARCHETYPES

Questions as 4 archetypes:
- Patient overview
- Conditions
- Medications
- Oncology timeline

### 1. QUESTION_TYPES and routing rules

This way, any user question is mapped onto a task type, even if it isn’t exactly one of your four eval questions.

Later, you can replace this with an LLM-powered classifier (see my googledoc notes too)

In [159]:
# more restricted version

QUESTION_TYPES = {
    "patient_overview": {
        "prompt_mode": "summary",
        "doc_types_primary": ["patient_overview"],
        "doc_types_med_fallback": ["medications"],          # new
        "doc_types_onco_fallback": ["oncology_timeline"],   # new
        "headings_primary": ["Recent Conditions", "Recent Results", "Procedures", "Medications"],
    },
    "conditions": {
        "prompt_mode": "extract_conditions",
        # Recent curated snapshot
        "doc_types_primary": ["patient_overview"],
        "headings_primary": ["Recent Conditions", "Recent Results"],
        # Longitudinal supplement from conditions.csv-derived chunks
        "doc_types_conditions_supplement": ["conditions"],
        "primary_num_results": 20,
        "conditions_num_results": 100,
    },
    "medications": {
        "prompt_mode": "extract_medications",
        "doc_types": ["patient_overview"],
        "headings": ["Medications"],
    },
    "oncology_timeline": {
        "description": "Oncology history and major events",
        "doc_types": ["oncology_timeline", "oncology_timeline_events"],
        "prompt_mode": "summarize_oncology_timeline",
    },
    # "current_status": {           # for “current status” questions
    # "description": "Most recent clinical status and events",
    # "doc_types": ["patient_overview", "encounters", "diagnostic_reports"],
    # "prompt_mode": "summarize_current_status",
    # },
}

In [160]:
# question-type classifier

def classify_question_type(question: str) -> str:
    q = question.lower()

    if any(word in q for word in ["overview", "summary", "background", "history"]):
        return "patient_overview"

    if any(word in q for word in ["condition", "diagnosis", "diagnosed"]):
        return "conditions"

    if any(word in q for word in ["medication", "drug", "therapy", "prescription"]):
        return "medications"

    if any(word in q for word in ["oncology", "cancer", "tumor", "chemo", "radiation", "stage"]):
        return "oncology_timeline"

    # fallback
    return "patient_overview"

### 4. Filtered context builder


In [161]:
from pathlib import Path
import sys

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "clinical_synopsis"))

import rag as rag_module

In [162]:
def build_filtered_context(search_results, question_type=None, headings=None):
    if headings:
        filtered = [
            doc for doc in search_results
            if doc.get("heading") in headings
        ]
        if filtered:
            return rag_module.build_context(filtered)

    # Fallback: if no filtered docs, use full results
    return rag_module.build_context(search_results)

### 3. Prompt mode builder

While above QUESTION_TYPES + context filtering decide what data the model sees (doc_types + headings) so that we enforce that answers are grounded in the curated summary rather than noisy raw data.. Now we define PROMPT_MODES / EXTRA snippets that will decide how to talk about that data (task framing + format).


BASE_INSTRUCTIONS: general behavior (be concise, don’t hallucinate, etc.).

EXTRA: archetype-specific behavior (conditions vs meds vs overview).

Context: the filtered patient_overview chunks.

Question: the user’s actual question.

E.g.
`prompt_mode="summary"` → use PATIENT_OVERVIEW_EXTRA to produce a concise overview.

`prompt_mode="extract_conditions"` → use CONDITIONS_EXTRA to do structured condition extraction.

`prompt_mode="extract_medications"` → use MEDICATIONS_EXTRA for meds.

`prompt_mode="summarize_oncology_timeline"` → use ONCOLOGY_TIMELINE_EXTRA for oncology history.

Thus, 
- `CONDITIONS_EXTRA` should be used when prompt_mode="extract_conditions".
- `PATIENT_OVERVIEW_EXTRA` should be used when prompt_mode="summary".

This is the structure below:
    - BASE_INSTRUCTIONS → general behavior.
    - PATIENT_OVERVIEW_EXTRA, CONDITIONS_EXTRA, MEDICATIONS_EXTRA, ONCOLOGY_TIMELINE_EXTRA → archetype-specific behavior.
    - PROMPT_MODES → maps prompt_mode to the right EXTRA snippet.

This was not enough:
```
PATIENT_OVERVIEW_EXTRA = """
For overview questions:

- Provide a concise summary of the patient’s medical background and current context.
- Focus on major conditions, treatments, and recent events.
- Base your summary strictly on the context; do NOT add conditions or events not in the context.

Format your answer in this structure:

1. **Summary**
   Provide 2–3 sentences summarizing the patient's overall medical background and current situation.

2. **Conditions**
   List the main diagnosed conditions with their status (active or resolved) and approximate dates, based on the "Recent Conditions" and "Recent Results" sections in the context.

3. **Medications**
   Summarize current or recent medications from the Medications section.

4. **Oncology timeline**
   Briefly describe the key oncology-related events (diagnosis, treatments, notable changes) in chronological order.

Do not add sections that are not supported by the context. If a section has no information in the context, you may omit it or say "No information documented."
"""
```

This was not enough
```MEDICATIONS_EXTRA = """
For questions about medications:

- Treat this as an extraction and summarization task based on the context.
- Focus on the patient's main *clinical* medications (e.g., oncology drugs, key chronic therapies),
  not over-the-counter or incidental mentions unless clearly clinically relevant.
- Use the Medications section from patient_overview.md as the primary source.
- List medications in chronological order by their documented start date (most recent first is acceptable).
- For each medication, preserve its status exactly (current, recent, or discontinued) as documented.
- Do NOT infer medications that are not mentioned.
- If a date is missing, say "date: not documented" instead of inventing one.

Format:
- One bullet per medication.
- Each bullet: **Medication name** — dose/regimen (if available); status; date: YYYY-MM-DD or "date: not documented".
"""
```

Are my instructions to long for a cheap model (like google seems to have suggested, unless I misread)?

Probably not. Each mode is sent separately, so `gpt-5.4-mini` receives `BASE_INSTRUCTIONS` plus **one** EXTRA block—not all four blocks at once. Your overview prompt is not unusually long; the observed behavior follows from instructions that are currently too broad or explicitly permit the wrong sources. Clear sections and concise, unambiguous rules are generally more important than making the prompt shorter. [developers.openai](https://developers.openai.com/api/docs/guides/prompt-engineering)

Why this output follows your prompt

Three lines in the current overview prompt explain the results:

| Current instruction | Consequence |
|---|---|
| “List the main diagnosed conditions” | Does not explicitly exclude stress, employment, or environmental findings, so the model includes them. |
| “Report only medications … current or active” | The acetaminophen, Percocet, fentanyl, and Oxycontin entries are active, so listing them is consistent with the instruction. |
| “oncology timeline or oncology-related conditions/medications” | Explicitly permits medication records to become oncology evidence, which caused the earlier opioid-as-oncology error. |

So this is primarily a **specification issue**, not a cheap-model or prompt-length issue.

Replace PATIENT_OVERVIEW_EXTRA

Use this more specific—but still shorter—version:

```python
PATIENT_OVERVIEW_EXTRA = """
For overview questions, use exactly these four sections and do not add others.

1. **Summary**
- Write exactly 3 sentences.
- Sentence 1: State the most important documented clinical conditions, including cancer history when documented in the oncology timeline.
- Sentence 2: State the current or recent clinically important status. Do not mention social, occupational, environmental, or administrative findings.
- Sentence 3: Give a one-sentence oncology synopsis using ONLY ONCOLOGY TIMELINE CONTEXT. Do not use medications as oncology evidence.

2. **Conditions**
- Include only documented clinical diagnoses and clinically meaningful comorbidities.
- Exclude social, occupational, environmental, administrative, and screening findings.
- Specifically exclude stress, employment status, not in labor force, and reports of violence in the environment.
- Do not reproduce every row from Recent Conditions.
- Preserve active/resolved status exactly.
- Format: **Condition** — status; date: YYYY-MM-DD.
- If none qualify, write: "No qualifying clinical conditions documented."

3. **Medications**
- Give a concise summary of medications explicitly documented as current or active.
- Do not list dose, route, strength, formulation, or duplicate ingredients.
- Group medications used for the same apparent purpose when documented together.
- For multiple active pain medicines, use one bullet named **Active analgesic regimen** and list only the medication names.
- Do not include completed, historical, inactive, or discontinued medications.
- If none are explicitly current or active, write exactly:
  "No current medication is documented in the provided medication snapshot."

4. **Oncology timeline**
- Use ONLY ONCOLOGY TIMELINE CONTEXT.
- Do not use patient_overview conditions, results, procedures, or medications as oncology evidence.
- Summarize diagnosis/staging, treatment episodes, and documented response or progression.
- Combine repeated treatment sessions and repeated identical response findings into date ranges.
- Do not infer remission, cure, recurrence, metastasis, or current cancer status.
- If no oncology timeline context is supplied, write:
  "No oncology timeline information documented."

Rules:
- Use only facts in the supplied context.
- Do not infer missing facts.
- Do not omit any of the four sections.
"""
```

Instead of (but this worked well for the previous patients!)
```
PATIENT_OVERVIEW_EXTRA = """
For overview questions:

You MUST follow this exact structure and formatting:

1. **Summary**
   - Write exactly 3 sentences.
   - In sentence 1, state the patient's key chronic or serious conditions (e.g., malignancies, embolism) with the date in brackets.
   - In sentence 2, state their overall status (e.g., recent events, resolved vs active issues).
   - In sentence 3, state their oncology treatment history. You MUST base the oncology history strictly on the oncology timeline context (oncology_timeline.md), not on inferred or invented details.

2. **Conditions**
   - List the main diagnosed conditions with their status (active or resolved) and approximate dates.
   - Use only information from the "Recent Conditions" and "Recent Results" sections in the context.
   - If there are no conditions in the context, write: "No condition information documented."

3. **Medications**
   - Report only medications explicitly documented as current or active in the Medications context.
   - Do NOT list medications marked completed, historical, inactive, or discontinued in this section.
   - If no medication is explicitly documented as current or active, write exactly:
     "No current medication is documented in the provided medication snapshot."
   - Historical cancer therapies may be described in the Oncology timeline section when supported by the oncology timeline context.

4. **Oncology timeline**
   - If oncology-related information is present in the context (oncology timeline or oncology-related conditions/medications), briefly describe the key oncology events in chronological order.
   - If there is no oncology-related information in the context, write: "No oncology-related information documented."

Rules:
- Base everything strictly on the context; do NOT invent conditions, medications, or events.
- Do not add extra sections or headings.
- Do not omit any of the four sections, even if they are empty; use the default sentence for empty sections.
"""
```

Expected Deeann output

With this prompt, the relevant overview portions should become:

```text
2. **Conditions**
- **Sprain of wrist** — active; date: 2022-05-31.
- **Preeclampsia** — resolved; date: 2017-05-01.

3. **Medications**
- **Active analgesic regimen** — acetaminophen, oxycodone, and fentanyl documented as active in 2022.

4. **Oncology timeline**
- **2013-10-11 to 2013-10-13** — breast cancer documented as active; biopsy and staging documented as T1N0, Stage 1/1A, HER2-positive.
- **2013-11-01 to 2013-12-17** — radiotherapy documented as completed.
- **2013-12-17 to 2017-07-18** — cancer disease progression repeatedly documented as improved.
```

Also change oncology mode

Your standalone `ONCOLOGY_TIMELINE_EXTRA` still explicitly says:

```text
- One bullet per oncology event.
```

That directly causes the list of every radiotherapy session. Replace only its `Format:` section with:

```python
Format:
- Combine repeated records of the same treatment into one treatment episode with a date range.
- Combine repeated identical response/progression records into one date-range bullet.
- Use 3–6 clinically meaningful bullets, in chronological order.
- Each bullet: **YYYY-MM-DD** or **YYYY-MM-DD to YYYY-MM-DD** — event type; concise factual description.
```

Keep the simpler design for now. First test this revised overview prompt across all nine patients; only move to a multi-call composition approach if the overview still fails to apply the clinical-condition and medication-grouping rules consistently.


#### INSTRUCTIONS

In [163]:
BASE_INSTRUCTIONS = rag_module.INSTRUCTIONS  # your current general instructions



PATIENT_OVERVIEW_EXTRA = """
For overview questions, use exactly these four sections and do not add others.

1. **Summary**
- Write exactly 3 sentences.
- Sentence 1: State the most important documented clinical conditions, including cancer history when documented in the oncology timeline.
- Sentence 2: State the current or recent clinically important status. Do not mention social, occupational, environmental, or administrative findings.
- Sentence 3: Give a one-sentence oncology synopsis using ONLY ONCOLOGY TIMELINE CONTEXT. Do not use medications as oncology evidence.

2. **Conditions**
- Include only documented clinical diagnoses and clinically meaningful comorbidities.
- Exclude social, occupational, environmental, administrative, and screening findings.
- Specifically exclude stress, employment status, not in labor force, and reports of violence in the environment.
- Do not reproduce every row from Recent Conditions.
- Preserve active/resolved status exactly.
- Format: **Condition** — status; date: YYYY-MM-DD.
- If none qualify, write: "No qualifying clinical conditions documented."

3. **Medications**
- Give a concise summary of medications explicitly documented as current or active.
- Do not list dose, route, strength, formulation, or duplicate ingredients.
- Group medications used for the same apparent purpose when documented together.
- For multiple active pain medicines, use one bullet named **Active analgesic regimen** and list only the medication names.
- Do not include completed, historical, inactive, or discontinued medications.
- If none are explicitly current or active, write exactly:
  "No current medication is documented in the provided medication snapshot."

4. **Oncology timeline**
- Use ONLY ONCOLOGY TIMELINE CONTEXT.
- Do not use patient_overview conditions, results, procedures, or medications as oncology evidence.
- Summarize diagnosis/staging, treatment episodes, and documented response or progression.
- Combine repeated treatment sessions and repeated identical response findings into date ranges.
- Do not infer remission, cure, recurrence, metastasis, or current cancer status.
- If no oncology timeline context is supplied, write:
  "No oncology timeline information documented."

Rules:
- Use only facts in the supplied context.
- Do not infer missing facts.
- Do not omit any of the four sections.
"""


CONDITIONS_EXTRA = """
For questions about diagnosed conditions:

- Use the supplied context only.
- Preserve each condition or finding's documented status and date.
- Do not infer diagnoses, status, dates, recurrence, remission, or causality.
- Separate diagnoses/disorders from findings according to the wording in the context.
- Entries labelled "(finding)" belong in Findings and social/functional history.
- Entries labelled "(disorder)" and documented diagnoses belong in Diagnoses and disorders.
- Within each section, list entries documented as active first, followed by resolved entries.
- Do not reorder entries further; retain their order from the supplied context within each status group.

Format your answer using exactly these two sections:

**Diagnoses and disorders**
- List documented diagnoses and disorders.
- Each bullet: **Name** — status; date: YYYY-MM-DD.
- If there are no documented diagnoses or disorders in the supplied context, write:
  "No diagnoses or disorders documented."

**Findings and social/functional history**
- List documented findings, including social, occupational, environmental, behavioral,
  and functional findings.
- Each bullet: **Name** — status; date: YYYY-MM-DD.
- If there are no documented findings in the supplied context, write:
  "No findings or social/functional history documented."
"""



MEDICATIONS_EXTRA = """
For questions about medications:

- Treat this as a medication-history extraction and prioritization task based only on the context.
- Use the Medications section from patient_overview.md as the primary source.
- Preserve each medication's documented status exactly. Do NOT describe a medication as current,
  active, ongoing, or discontinued unless that status is explicitly documented.
- If all listed medications are historical or marked completed, explicitly state:
  "No current medication is documented in the provided medication snapshot."
- Prioritize clinically significant therapies over routine, duplicate, short-term, or remote medications.
- In an oncology patient, prioritize documented antineoplastic or endocrine cancer therapies.
- Group all documented historical hormonal contraceptive therapies into exactly one bullet named
  "**Historical hormonal contraception**." This includes oral contraceptives, transdermal contraceptive
  patches, and contraceptive implants when they appear in the context.
- Do NOT create separate bullets for individual historical contraceptive products after grouping them.
- Exclude one-off symptomatic or short-course medications (for example, cold/flu, cough, pain, or
  sleep products) when other clinically significant medication history is present.
- Do NOT list duplicate historical entries for the same medication or medication class. Retain only
  the most recent documented date within a grouped bullet.
- List antineoplastic, endocrine cancer therapy, or other disease-modifying therapy as separate
  medication bullets, even if marked completed.
- Do NOT infer indication, current use, dose, regimen, treatment response, or clinical importance
  beyond what is documented.
- If a date is missing, write "date: not documented"; do not invent one.

Format:
- Begin with one status sentence:
  - If no medication is explicitly current/active: "No current medication is documented in the provided medication snapshot."
  - Otherwise: "Current/recent medications documented in the provided snapshot:"
- Then use one bullet per medication or clinically coherent medication group.
- Each bullet: **Medication or group** — documented status; date or date range in YYYY-MM-DD format; brief factual description only when supported by the context.
- Convert documented timestamps to their calendar date only. Do not include a time, time zone, or infer a date that is not documented.
"""

ONCOLOGY_TIMELINE_EXTRA = """
For questions about oncology history:

- Treat this as an extraction and summarization task based on the context.
- Focus on the patient's main *oncology-related* events (e.g., diagnoses, staging, treatments, progression or response),
  not unrelated conditions or encounters.
- Use oncology-related sections (e.g., oncology_timeline chunks and relevant parts of patient_overview.md) as the primary source.
- List events in strict chronological order by their documented date (earliest first).
- For each event, preserve its type and status exactly as documented (diagnosis, treatment start, progression, response, etc.).
- Do NOT infer oncology events that are not mentioned.
- If a date is missing, say "date: not documented" instead of inventing one.

Format:
- Combine repeated records of the same treatment into one treatment episode with a date range.
- Combine repeated identical response/progression records into one date-range bullet.
- Use 3–6 clinically meaningful bullets, in chronological order.
- Each bullet: **YYYY-MM-DD** or **YYYY-MM-DD to YYYY-MM-DD** — event type; concise factual description.
"""



PROMPT_MODES = {
    "summary": PATIENT_OVERVIEW_EXTRA,
    "extract_conditions": CONDITIONS_EXTRA,
    "extract_medications": MEDICATIONS_EXTRA,
    "summarize_oncology_timeline": ONCOLOGY_TIMELINE_EXTRA,
}

In [164]:
def build_prompt_with_mode(question, context, prompt_mode):
    extra = PROMPT_MODES.get(prompt_mode, "")
    system_instructions = BASE_INSTRUCTIONS + "\n\n" + extra

    prompt = f"""
{system_instructions}

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
""".strip()

    return prompt

### (1) rag_new

In [165]:
# def rag_new(
#     query,
#     patient_id,
#     question_type=None,
#     num_results=5,
#     model="gpt-5.4-mini",
#     search_type="hybrid",
# ):
#     cfg = QUESTION_TYPES.get(question_type, {})
#     prompt_mode = cfg.get("prompt_mode", "default")
#     doc_types = cfg.get("doc_types", None)
#     headings = cfg.get("headings", None)

#     # 1. Retrieval, already constrained by doc_types
#     search_results = rag_module.search(
#         query=query,
#         patient_id=patient_id,
#         doc_types=doc_types,
#         is_oncology=None,
#         num_results=num_results,
#         search_type=search_type,
#     )

#     # 2. Heading-level filter into context
#     context = build_filtered_context(
#         search_results,
#         question_type=question_type,
#         headings=headings,
#     )

#     # 3. Prompt with template
#     prompt = build_prompt_with_mode(
#         question=query,
#         context=context,
#         prompt_mode=prompt_mode,
#     )

#     llm_out = rag_module.llm(prompt=prompt, model=model)
#     answer = llm_out["answer"]
#     token_stats = llm_out["token_stats"]
#     cost_info = calculate_openai_cost(model, token_stats)

#     return {
#         "answer": answer,
#         **token_stats,
#         **cost_info,
#     }

In [166]:
# def rag_new(
#     query,
#     patient_id,
#     question_type=None,
#     num_results=5,
#     model="gpt-5.4-mini",
#     search_type="hybrid",
# ):
#     cfg = QUESTION_TYPES.get(question_type, {})
#     prompt_mode = cfg.get("prompt_mode", "summary")
#     doc_types = cfg.get("doc_types", None)
#     headings = cfg.get("headings", None)

#     # 1. Retrieval, constrained by doc_types if present
#     search_results = rag_module.search(
#         query=query,
#         patient_id=patient_id,
#         doc_types=doc_types,
#         is_oncology=None,
#         num_results=num_results,
#         search_type=search_type,
#     )

#     # 2. Filter context by headings / question_type
#     context = build_filtered_context(search_results, question_type=question_type, headings=headings)

#     # 3. Build prompt with correct mode
#     prompt = build_prompt_with_mode(
#         question=query,
#         context=context,
#         prompt_mode=prompt_mode,
#     )

#     # 4. Call LLM, compute cost, etc.
#     llm_out = rag_module.llm(prompt=prompt, model=model)
#     answer = llm_out["answer"]
#     token_stats = llm_out["token_stats"]
#     cost_info = rag_module.calculate_openai_cost(model, token_stats)

#     return {
#         "answer": answer,
#         **token_stats,
#         **cost_info,
#     }

In [167]:
# # the version that doesn't get medications from patient_overview
# def rag_new(
#     query,
#     patient_id,
#     question_type=None,
#     num_results=5,
#     model="gpt-5.4-mini",
#     search_type="hybrid",
# ):
#     cfg = QUESTION_TYPES.get(question_type, {})
#     prompt_mode = cfg.get("prompt_mode", "summary")
#     doc_types = cfg.get("doc_types", None) or cfg.get("doc_types_primary")
#     headings = cfg.get("headings", None) or cfg.get("headings_primary")

#     # 1. Retrieval
#     if search_type == "lexical":
#         search_results = rag_module.search(
#             query=query,
#             patient_id=patient_id,
#             doc_types=doc_types,
#             is_oncology=None,
#             num_results=num_results,
#         )
#     elif search_type == "semantic":
#         search_results = rag_module.semantic_search(
#             query=query,
#             patient_id=patient_id,
#             doc_types=doc_types,
#             is_oncology=None,
#             num_results=num_results,
#         )
#     elif search_type == "hybrid":
#         search_results = rag_module.hybrid_search(
#             query=query,
#             patient_id=patient_id,
#             doc_types=doc_types,
#             is_oncology=None,
#             num_results=num_results,
#         )
#     else:
#         raise ValueError("search_type must be one of: lexical, semantic, hybrid")

#     # 2. Filter context by headings / question_type
#     context = build_filtered_context(
#         search_results,
#         question_type=question_type,
#         headings=headings,
#     )

#     # 3. Build prompt
#     prompt = build_prompt_with_mode(
#         question=query,
#         context=context,
#         prompt_mode=prompt_mode,
#     )

#     # 4. Call LLM
#     llm_out = rag_module.llm(prompt=prompt, model=model)
#     answer = llm_out["answer"]
#     token_stats = llm_out["token_stats"]
#     cost_info = rag_module.calculate_openai_cost(model, token_stats)

#     return {
#         "answer": answer,
#         **token_stats,
#         **cost_info,
#     }

### (2) rag_new

In [168]:
# # def _extract_patient_name(search_results):
# #     for doc in search_results:
# #         heading = doc.get("heading", "")
# #         if heading.startswith("Patient Overview:"):
# #             return heading.split(":", 1)[1].strip()
# #     return None

# # def _extract_patient_name(search_results):
# #     for doc in search_results:
# #         title = doc.get("title", "")
# #         if title.startswith("Patient Overview:"):
# #             return title.split(":", 1)[1].strip()
# #     return None

# from datetime import date
# import re

# def _calculate_age(dob_str, as_of=None):
#     if not dob_str:
#         return None
#     as_of = as_of or date.today()
#     dob = date.fromisoformat(dob_str[:10])
#     return as_of.year - dob.year - ((as_of.month, as_of.day) < (dob.month, dob.day))

# def _extract_patient_identity(search_results):
#     patient_name = None
#     patient_dob = None
#     patient_gender = None

#     for doc in search_results:
#         title = doc.get("title", "")
#         if title.startswith("Patient Overview:"):
#             patient_name = title.split(":", 1)[1].strip()

#         chunk_text = doc.get("chunk_text", "")
#         dob_match = re.search(r"^\s*-\s*Birth date:\s*(\d{4}-\d{2}-\d{2})", chunk_text, re.M)
#         gender_match = re.search(r"^\s*-\s*Gender:\s*([A-Za-z]+)", chunk_text, re.M)

#         if dob_match:
#             patient_dob = dob_match.group(1)
#         if gender_match:
#             patient_gender = gender_match.group(1)

#     patient_age = _calculate_age(patient_dob)
#     return patient_name, patient_dob, patient_age, patient_gender


# def _search(search_type, query, patient_id, doc_types, num_results):
#     if search_type == "lexical":
#         return rag_module.search(
#             query=query,
#             patient_id=patient_id,
#             doc_types=doc_types,
#             is_oncology=None,
#             num_results=num_results,
#         )
#     if search_type == "semantic":
#         return rag_module.semantic_search(
#             query=query,
#             patient_id=patient_id,
#             doc_types=doc_types,
#             is_oncology=None,
#             num_results=num_results,
#         )
#     if search_type == "hybrid":
#         return rag_module.hybrid_search(
#             query=query,
#             patient_id=patient_id,
#             doc_types=doc_types,
#             is_oncology=None,
#             num_results=num_results,
#         )
#     raise ValueError("search_type must be one of: lexical, semantic, hybrid")


# def _has_heading(docs, heading):
#     return any(doc.get("heading") == heading for doc in docs)


# def _dedupe_by_chunk_id(docs):
#     seen = set()
#     merged = []
#     for doc in docs:
#         chunk_id = doc.get("chunk_id")
#         if chunk_id in seen:
#             continue
#         seen.add(chunk_id)
#         merged.append(doc)
#     return merged


# def rag_new(
#     query,
#     patient_id,
#     question_type=None,
#     num_results=5,
#     model="gpt-5.4-mini",
#     search_type="hybrid",
# ):
#     cfg = QUESTION_TYPES.get(question_type, {})
#     prompt_mode = cfg.get("prompt_mode", "summary")
#     doc_types = cfg.get("doc_types", None) or cfg.get("doc_types_primary")
#     headings = cfg.get("headings", None) or cfg.get("headings_primary")

#     search_results = _search(
#         search_type=search_type,
#         query=query,
#         patient_id=patient_id,
#         doc_types=doc_types,
#         num_results=num_results,
#     )

#     if question_type == "patient_overview" and not _has_heading(search_results, "Medications"):
#         medication_results = _search(
#             search_type=search_type,
#             query="medications",
#             patient_id=patient_id,
#             doc_types=["patient_overview"],
#             num_results=max(num_results, 10),
#         )
#         medication_results = [
#             doc for doc in medication_results
#             if doc.get("heading") == "Medications"
#         ]
#         search_results = _dedupe_by_chunk_id(search_results + medication_results)

#     context = build_filtered_context(
#         search_results,
#         question_type=question_type,
#         headings=headings,
#     )

#     prompt = build_prompt_with_mode(
#         question=query,
#         context=context,
#         prompt_mode=prompt_mode,
#     )

#     llm_out = rag_module.llm(prompt=prompt, model=model)
#     answer = llm_out["answer"]
#     token_stats = llm_out["token_stats"]
#     cost_info = rag_module.calculate_openai_cost(model, token_stats)
#     patient_name, patient_dob, patient_age, patient_gender = _extract_patient_identity(search_results)

#     return {
#         "patient_id": patient_id,
#         "patient_name": patient_name,
#         "patient_dob": patient_dob,
#         "patient_age_years": patient_age,
#         "patient_gender": patient_gender,
#         "answer": answer,
#         **token_stats,
#         **cost_info,
#     }

### (3) rag_new

In [169]:
# from datetime import date
# import re

# def _calculate_age(dob_str, as_of=None):
#     if not dob_str:
#         return None
#     as_of = as_of or date.today()
#     dob = date.fromisoformat(dob_str[:10])
#     return as_of.year - dob.year - ((as_of.month, as_of.day) < (dob.month, dob.day))

# def _extract_patient_identity(search_results):
#     patient_name = None
#     patient_dob = None
#     patient_gender = None

#     for doc in search_results:
#         title = doc.get("title", "")
#         if title.startswith("Patient Overview:"):
#             patient_name = title.split(":", 1)[1].strip()

#         chunk_text = doc.get("chunk_text", "")
#         dob_match = re.search(r"^\s*-\s*Birth date:\s*(\d{4}-\d{2}-\d{2})", chunk_text, re.M)
#         gender_match = re.search(r"^\s*-\s*Gender:\s*([A-Za-z]+)", chunk_text, re.M)

#         if dob_match:
#             patient_dob = dob_match.group(1)
#         if gender_match:
#             patient_gender = gender_match.group(1)

#     patient_age = _calculate_age(patient_dob)
#     return patient_name, patient_dob, patient_age, patient_gender


# def _search(search_type, query, patient_id, doc_types, num_results):
#     if search_type == "lexical":
#         return rag_module.search(
#             query=query,
#             patient_id=patient_id,
#             doc_types=doc_types,
#             is_oncology=None,
#             num_results=num_results,
#         )
#     if search_type == "semantic":
#         return rag_module.semantic_search(
#             query=query,
#             patient_id=patient_id,
#             doc_types=doc_types,
#             is_oncology=None,
#             num_results=num_results,
#         )
#     if search_type == "hybrid":
#         return rag_module.hybrid_search(
#             query=query,
#             patient_id=patient_id,
#             doc_types=doc_types,
#             is_oncology=None,
#             num_results=num_results,
#         )
#     raise ValueError("search_type must be one of: lexical, semantic, hybrid")


# def _has_heading(docs, heading):
#     return any(doc.get("heading") == heading for doc in docs)


# def _dedupe_by_chunk_id(docs):
#     seen = set()
#     merged = []
#     for doc in docs:
#         chunk_id = doc.get("chunk_id")
#         if chunk_id in seen:
#             continue
#         seen.add(chunk_id)
#         merged.append(doc)
#     return merged


# def rag_new(
#     query,
#     patient_id,
#     question_type=None,
#     num_results=5,
#     model="gpt-5.4-mini",
#     search_type="hybrid",
# ):
#     cfg = QUESTION_TYPES.get(question_type, {})
#     prompt_mode = cfg.get("prompt_mode", "summary")

#     primary_doc_types = cfg.get("doc_types") or cfg.get("doc_types_primary")
#     primary_headings = cfg.get("headings") or cfg.get("headings_primary")

#     # 1) Primary overview retrieval
#     overview_results = _search(
#         search_type=search_type,
#         query=query,
#         patient_id=patient_id,
#         doc_types=primary_doc_types,
#         num_results=num_results,
#     )

#     # keep your medication fallback
#     if question_type == "patient_overview" and not _has_heading(overview_results, "Medications"):
#         medication_results = _search(
#             search_type=search_type,
#             query="medications",
#             patient_id=patient_id,
#             doc_types=["patient_overview"],
#             num_results=max(num_results, 10),
#         )
#         medication_results = [d for d in medication_results if d.get("heading") == "Medications"]
#         overview_results = _dedupe_by_chunk_id(overview_results + medication_results)

#     # 2) Filter only allowed overview headings
#     if primary_headings:
#         overview_filtered = [d for d in overview_results if d.get("heading") in primary_headings]
#         if not overview_filtered:
#             overview_filtered = overview_results
#     else:
#         overview_filtered = overview_results

#     # 3) Always add oncology timeline for patient_overview
#     oncology_results = []
#     if question_type == "patient_overview":
#         oncology_results = _search(
#             search_type=search_type,
#             query="Summarize the patient's oncology history, treatments, and documented response or progression.",
#             patient_id=patient_id,
#             doc_types=["oncology_timeline"],
#             num_results=20,
#         )

#     # 4) Merge and build final context
#     all_results = _dedupe_by_chunk_id(overview_filtered + oncology_results)
#     context = rag_module.build_context(all_results)

#     prompt = build_prompt_with_mode(
#         question=query,
#         context=context,
#         prompt_mode=prompt_mode,
#     )

#     llm_out = rag_module.llm(prompt=prompt, model=model)
#     answer = llm_out["answer"]
#     token_stats = llm_out["token_stats"]
#     cost_info = rag_module.calculate_openai_cost(model, token_stats)
#     patient_name, patient_dob, patient_age, patient_gender = _extract_patient_identity(overview_results)

#     return {
#         "patient_id": patient_id,
#         "patient_name": patient_name,
#         "patient_dob": patient_dob,
#         "patient_age_years": patient_age,
#         "patient_gender": patient_gender,
#         "answer": answer,
#         **token_stats,
#         **cost_info,
#     }

# (4) rag_new with fallback for conditions


In [170]:
from datetime import date
import re

def _calculate_age(dob_str, as_of=None):
    if not dob_str:
        return None
    as_of = as_of or date.today()
    dob = date.fromisoformat(dob_str[:10])
    return as_of.year - dob.year - ((as_of.month, as_of.day) < (dob.month, dob.day))

def _extract_patient_identity(search_results):
    patient_name = None
    patient_dob = None
    patient_gender = None

    for doc in search_results:
        title = doc.get("title", "")
        if title.startswith("Patient Overview:"):
            patient_name = title.split(":", 1)[1].strip()

        chunk_text = doc.get("chunk_text", "")
        dob_match = re.search(r"^\s*-\s*Birth date:\s*(\d{4}-\d{2}-\d{2})", chunk_text, re.M)
        gender_match = re.search(r"^\s*-\s*Gender:\s*([A-Za-z]+)", chunk_text, re.M)

        if dob_match:
            patient_dob = dob_match.group(1)
        if gender_match:
            patient_gender = gender_match.group(1)

    patient_age = _calculate_age(patient_dob)
    return patient_name, patient_dob, patient_age, patient_gender


def _search(search_type, query, patient_id, doc_types, num_results):
    if search_type == "lexical":
        return rag_module.search(
            query=query,
            patient_id=patient_id,
            doc_types=doc_types,
            is_oncology=None,
            num_results=num_results,
        )
    if search_type == "semantic":
        return rag_module.semantic_search(
            query=query,
            patient_id=patient_id,
            doc_types=doc_types,
            is_oncology=None,
            num_results=num_results,
        )
    if search_type == "hybrid":
        return rag_module.hybrid_search(
            query=query,
            patient_id=patient_id,
            doc_types=doc_types,
            is_oncology=None,
            num_results=num_results,
        )
    raise ValueError("search_type must be one of: lexical, semantic, hybrid")


def _has_heading(docs, heading):
    return any(doc.get("heading") == heading for doc in docs)


def _dedupe_by_chunk_id(docs):
    seen = set()
    merged = []
    for doc in docs:
        chunk_id = doc.get("chunk_id")
        if chunk_id in seen:
            continue
        seen.add(chunk_id)
        merged.append(doc)
    return merged


def rag_new(
    query,
    patient_id,
    question_type=None,
    num_results=5,
    model="gpt-5.4-mini",
    search_type="hybrid",
):
    cfg = QUESTION_TYPES.get(question_type, {})
    prompt_mode = cfg.get("prompt_mode", "summary")

    primary_doc_types = cfg.get("doc_types") or cfg.get("doc_types_primary")
    primary_headings = cfg.get("headings") or cfg.get("headings_primary")
    primary_num_results = cfg.get("primary_num_results", num_results)

    # 1) Primary retrieval
    overview_results = _search(
        search_type=search_type,
        query=query,
        patient_id=patient_id,
        doc_types=primary_doc_types,
        num_results=primary_num_results,
    )

    # 1a) Medication heading fallback for patient overview
    if (
        question_type == "patient_overview"
        and not _has_heading(overview_results, "Medications")
    ):
        medication_results = _search(
            search_type=search_type,
            query="medications",
            patient_id=patient_id,
            doc_types=["patient_overview"],
            num_results=max(num_results, 10),
        )

        medication_results = [
            d for d in medication_results
            if d.get("heading") == "Medications"
        ]

        overview_results = _dedupe_by_chunk_id(
            overview_results + medication_results
        )

    # 2) Apply permitted headings to the primary overview source
    if primary_headings:
        overview_filtered = [
            d for d in overview_results
            if d.get("heading") in primary_headings
        ]
        if not overview_filtered:
            overview_filtered = overview_results
    else:
        overview_filtered = overview_results

    # 3) Always add longitudinal conditions.csv-derived rows
    #    for the dedicated Conditions question.
    conditions_results = []
    if question_type == "conditions":
        conditions_results = _search(
            search_type=search_type,
            query=query,
            patient_id=patient_id,
            doc_types=cfg.get(
                "doc_types_conditions_supplement",
                ["conditions"],
            ),
            num_results=cfg.get("conditions_num_results", 100),
        )
        conditions_results = _dedupe_by_chunk_id(conditions_results)

    # 4) Always add oncology timeline for patient overview
    oncology_results = []
    if question_type == "patient_overview":
        oncology_results = _search(
            search_type=search_type,
            query=(
                "Summarize the patient's oncology history, treatments, "
                "and documented response or progression."
            ),
            patient_id=patient_id,
            doc_types=cfg.get(
                "doc_types_onco_fallback",
                ["oncology_timeline"],
            ),
            num_results=20,
        )
        oncology_results = _dedupe_by_chunk_id(oncology_results)

    # 5) Build source-labelled context
    if question_type == "conditions":
        recent_context = rag_module.build_context(overview_filtered)
        longitudinal_context = rag_module.build_context(conditions_results)

        context = f"""
RECENT PATIENT OVERVIEW CONDITIONS:
{recent_context}

LONGITUDINAL CONDITIONS RECORDS:
{longitudinal_context}
""".strip()

    elif question_type == "patient_overview":
        overview_context = rag_module.build_context(overview_filtered)
        oncology_context = rag_module.build_context(oncology_results)

        context = f"""
PATIENT OVERVIEW CONTEXT:
{overview_context}

ONCOLOGY TIMELINE CONTEXT:
{oncology_context}
""".strip()

    else:
        all_results = _dedupe_by_chunk_id(overview_filtered + oncology_results)
        context = rag_module.build_context(all_results)

    # 6) Prompt and LLM call
    prompt = build_prompt_with_mode(
        question=query,
        context=context,
        prompt_mode=prompt_mode,
    )

    llm_out = rag_module.llm(prompt=prompt, model=model)
    answer = llm_out["answer"]
    token_stats = llm_out["token_stats"]
    cost_info = rag_module.calculate_openai_cost(model, token_stats)

    patient_name, patient_dob, patient_age, patient_gender = (
        _extract_patient_identity(overview_results)
    )

    return {
        "patient_id": patient_id,
        "patient_name": patient_name,
        "patient_dob": patient_dob,
        "patient_age_years": patient_age,
        "patient_gender": patient_gender,
        "answer": answer,
        **token_stats,
        **cost_info,
    }

# Pick a few questions per archetype that you’ll compare manually:



In [171]:
selected_patient_ids

['d65197b3-056a-2136-b584-77f43c29da3f',
 'f3739580-797d-ae04-eebf-aeddb2fc2f64',
 '4736727e-63f4-071a-1516-a49310f5a052',
 '29f6beee-162f-0113-7884-72245814693f',
 '41681ed6-efc5-94c0-1bc0-f60b34dbd31b',
 'aee216e6-cbe8-eaf2-3241-4bd1e8a01494',
 'ecc4a7d0-8838-36b4-44ba-676d5a1f7927',
 '3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678',
 'f203e11d-5573-1624-69b8-af8436987b3e']

In [172]:
# # Overview IN THIS VERSION FOR COMPLEX PATIENTS INFO is missing because it's not in patient_overview.md

# overview_out = rag_new(
#     query=overview_question,
#     patient_id=patient_id,
#     question_type="patient_overview",
#     num_results=5,
#     model="gpt-5.4-mini",
#     search_type="hybrid",
# )

# print(f"=== Patient overview answer for patient {patient_id} ===")
# print(overview_out["answer"])

# # Conditions
# conditions_out = rag_new(
#     query=conditions_question,
#     patient_id=patient_id,
#     question_type="conditions",
#     num_results=5,
#     model="gpt-5.4-mini",
#     search_type="hybrid",
# )

# print(f"\n=== Conditions answer for patient {patient_id} ===")
# print(conditions_out["answer"])

# # Medications
# medications_out = rag_new(
#     query=medications_question,
#     patient_id=patient_id,
#     question_type="medications",
#     num_results=5,
#     model="gpt-5.4-mini",
#     search_type="hybrid",
# )

# print(f"\n=== Medications answer for patient {patient_id} ===")
# print(medications_out["answer"])

# # Oncology timeline (if configured)
# oncology_out = rag_new(
#     query=oncology_question,
#     patient_id=patient_id,
#     question_type="oncology_timeline",
#     num_results=5,
#     model="gpt-5.4-mini",
#     search_type="hybrid",
# )

# print(f"\n=== Oncology timeline answer for patient {patient_id} ===")
# print(oncology_out["answer"])

In [173]:
# # Overview 2nd version of QUESTION_TYPES with         "headings_primary": ["Recent Conditions", "Recent Results", "Procedures", "Medications"],

# overview_out = rag_new(
#     query=overview_question,
#     patient_id=patient_id,
#     question_type="patient_overview",
#     num_results=5,
#     model="gpt-5.4-mini",
#     search_type="hybrid",
# )

# print(f"=== Patient overview answer for patient {patient_id} ===")
# print(overview_out["answer"])

# # Conditions
# conditions_out = rag_new(
#     query=conditions_question,
#     patient_id=patient_id,
#     question_type="conditions",
#     num_results=5,
#     model="gpt-5.4-mini",
#     search_type="hybrid",
# )

# print(f"\n=== Conditions answer for patient {patient_id} ===")
# print(conditions_out["answer"])

# # Medications
# medications_out = rag_new(
#     query=medications_question,
#     patient_id=patient_id,
#     question_type="medications",
#     num_results=5,
#     model="gpt-5.4-mini",
#     search_type="hybrid",
# )

# print(f"\n=== Medications answer for patient {patient_id} ===")
# print(medications_out["answer"])

# # Oncology timeline (if configured)
# oncology_out = rag_new(
#     query=oncology_question,
#     patient_id=patient_id,
#     question_type="oncology_timeline",
#     num_results=5,
#     model="gpt-5.4-mini",
#     search_type="hybrid",
# )

# print(f"\n=== Oncology timeline answer for patient {patient_id} ===")
# print(oncology_out["answer"])

Turns out that **“Recent” = top 10 diagnostic reports.**, so now I see what needs to be fixed for complex patients (with lots of records), of which we have 3 in the set of 9. For them patient_overview.md is an incomplete source for the summary, thus with our current pre-filtering we get:

3. **Medications**
No information documented.

4. **Oncology timeline**
No oncology-related information documented.

We need to add as fallback looking at medications.csv (latest n records with dates and a link for more) and oncology_timeline.

In [174]:
# # Confirm that "Medications" appears in the set of headings for this patient

# results = rag_module.search(
#     query="Provide an overview of this patient.",
#     patient_id="f203e11d-5573-1624-69b8-af8436987b3e",  # or one of the complex patients
#     doc_types=["patient_overview"],
#     num_results=50,
#     search_type="hybrid",
# )

# {doc.get("heading") for doc in results}

In [175]:
# # Confirm that "Medications" appears in the set of headings for this patient
# results = rag_module.hybrid_search(
#     query="Provide an overview of this patient.",
#     patient_id="f203e11d-5573-1624-69b8-af8436987b3e",
#     doc_types=["patient_overview"],
#     is_oncology=None,
#     num_results=50,
# )

In [176]:
# {doc.get("heading") for doc in results}

In [177]:
# QUESTION_TYPES["patient_overview"]


In [178]:
# headings = QUESTION_TYPES["patient_overview"].get("headings") or QUESTION_TYPES["patient_overview"].get("headings_primary")
# context = build_filtered_context(
#     search_results=results,
#     question_type="patient_overview",
#     headings=headings,
# )

# print(context)

# f203e11d-5573-1624-69b8-af8436987b3e Shawana711 Lakin515

In [179]:
patient_id = "f203e11d-5573-1624-69b8-af8436987b3e" #high complexity patient

overview_question = "Provide a brief overview of this patient's medical background and current status."
conditions_question = "What are this patient's main diagnosed conditions and their status?"
medications_question = "What medications is this patient currently or recently taking?"
oncology_question = "Summarize this patient's oncology history."

In [180]:
# Overview 3rd version with new PATIENT_OVERVIEW_EXTRA

overview_out = rag_new(
    query=overview_question,
    patient_id=patient_id,
    question_type="patient_overview",
    num_results=5,
    model="gpt-5.4-mini",
    search_type="hybrid",
)

print(
    f"=== Patient overview answer for {overview_out['patient_name']} "
    f"(DOB {overview_out['patient_dob']}, Age: {overview_out['patient_age_years']} years, "
    f"Gender: {overview_out['patient_gender']}) ==="
)
print(overview_out["answer"])

# Conditions
conditions_out = rag_new(
    query=conditions_question,
    patient_id=patient_id,
    question_type="conditions",
    num_results=5,
    model="gpt-5.4-mini",
    search_type="hybrid",
)

print(f"\n=== Conditions answer for patient {conditions_out['patient_name']} ===")
print(conditions_out["answer"])

# Medications
medications_out = rag_new(
    query=medications_question,
    patient_id=patient_id,
    question_type="medications",
    num_results=5,
    model="gpt-5.4-mini",
    search_type="hybrid",
)

print(f"\n=== Medications answer for patient {medications_out['patient_name']} ===")
print(medications_out["answer"])

# Oncology timeline (if configured)
oncology_out = rag_new(
    query=oncology_question,
    patient_id=patient_id,
    question_type="oncology_timeline",
    num_results=5,
    model="gpt-5.4-mini",
    search_type="hybrid",
)

print(f"\n=== Oncology timeline answer for patient {oncology_out['patient_name']} ===")
print(oncology_out["answer"])

=== Patient overview answer for Shawana711 Lakin515 (DOB 1965-01-05, Age: 61 years, Gender: female) ===
1. **Summary**
- The documented clinical conditions include hypoxemia, acute pulmonary embolism, pneumonia, sepsis caused by virus, viral sinusitis, and concussion with loss of consciousness; oncology history includes acute myeloid leukemia and breast cancer.
- The most recent clinically important status is hypoxemia marked active on 2020-05-04 and concussion with loss of consciousness resolved on 2022-04-22.
- Oncology timeline shows active breast cancer diagnosed in 2015 with T2N0, Stage 2/2A, HER2-negative disease and repeated completed radiotherapy course events from 2015, with subsequent observations that the patient’s condition improved through 2019.

2. **Conditions**
- **Hypoxemia** — active; date: 2020-05-04.
- **Acute pulmonary embolism** — resolved; date: 2020-05-11.
- **Pneumonia** — resolved; date: 2020-05-04.
- **Sepsis caused by virus** — resolved; date: 2020-05-04.
- 

In [181]:
# # Conditions - SEMANTIC SEARCH
# conditions_out = rag_new(
#     query=conditions_question,
#     patient_id=patient_id,
#     question_type="conditions",
#     num_results=5,
#     model="gpt-5.4-mini",
#     search_type="semantic",
# )

# print(f"\n=== Conditions answer for patient {conditions_out['patient_name']} ===")
# print(conditions_out["answer"])

# # Medications  - SEMANTIC SEARCH
# medications_out = rag_new(
#     query=medications_question,
#     patient_id=patient_id,
#     question_type="medications",
#     num_results=5,
#     model="gpt-5.4-mini",
#     search_type="semantic",
# )

# print(f"\n=== Medications answer for patient {medications_out['patient_name']} ===")
# print(medications_out["answer"])


In [182]:
# docs = _search(
#     search_type="hybrid",
#     query=overview_question,
#     patient_id=patient_id,
#     doc_types=["patient_overview"],
#     num_results=5,
# )

# doc = docs[0]
# print("TITLE:", doc.get("title"))
# print("HEADING:", doc.get("heading"))
# print("CHUNK TEXT:")
# print(doc.get("chunk_text", ""))

In [183]:
# print(_extract_patient_identity(docs))

In [184]:
# doc = docs[0]
# chunk_text = doc.get("chunk_text", "")

# print("TITLE:", doc.get("title"))
# print("CHUNK TEXT:")
# print(chunk_text)

# print("LINE CHECK:")
# for line in chunk_text.splitlines():
#     print(repr(line))

In [185]:
# # test extractions separately
# from datetime import date
# import re

# title = doc.get("title", "")
# patient_name = title.split(":", 1)[1].strip() if title.startswith("Patient Overview:") else None

# dob_match = re.search(r"^\s*-\s*Birth date:\s*(\d{4}-\d{2}-\d{2})", chunk_text, re.M)
# gender_match = re.search(r"^\s*-\s*Gender:\s*([A-Za-z]+)", chunk_text, re.M)

# patient_dob = dob_match.group(1) if dob_match else None
# patient_gender = gender_match.group(1) if gender_match else None

# def calculate_age(dob_str, as_of=None):
#     if not dob_str:
#         return None
#     as_of = as_of or date.today()
#     dob = date.fromisoformat(dob_str)
#     return as_of.year - dob.year - ((as_of.month, as_of.day) < (dob.month, dob.day))

# patient_age = calculate_age(patient_dob)

# print("patient_name:", patient_name)
# print("patient_dob:", patient_dob)
# print("patient_age:", patient_age)
# print("patient_gender:", patient_gender)

In [186]:
# doc = docs[0]
# print(doc.keys())
# print(doc.get("heading"))
# print(doc.get("title"))
# print(doc.get("chunk_text", "")[:300])

In [187]:
# # Confirm that "Medications" appears in the set of headings for this patient
# results = rag_module.hybrid_search(
#     query="Provide an overview of this patient.",
#     patient_id="f203e11d-5573-1624-69b8-af8436987b3e",
#     doc_types=["patient_overview"],
#     is_oncology=None,
#     num_results=50,
# )
# {doc.get("heading") for doc in results}

In [188]:
# QUESTION_TYPES["patient_overview"]


In [189]:
# headings = QUESTION_TYPES["patient_overview"].get("headings") or QUESTION_TYPES["patient_overview"].get("headings_primary")
# context = build_filtered_context(
#     search_results=results,
#     question_type="patient_overview",
#     headings=headings,
# )

# print(context)

# d65197b3-056a-2136-b584-77f43c29da3f

In [190]:
patient_id = "d65197b3-056a-2136-b584-77f43c29da3f" #low complexity patient

overview_question = "Provide a brief overview of this patient's medical background and current status."
conditions_question = "What are this patient's main diagnosed conditions and their status?"
medications_question = "What medications is this patient currently or recently taking?"
oncology_question = "Summarize this patient's oncology history."

In [191]:
# Overview 3rd version with new PATIENT_OVERVIEW_EXTRA

overview_out = rag_new(
    query=overview_question,
    patient_id=patient_id,
    question_type="patient_overview",
    num_results=5,
    model="gpt-5.4-mini",
    search_type="hybrid",
)

print(
    f"=== Patient overview answer for {overview_out['patient_name']} "
    f"(DOB {overview_out['patient_dob']}, Age: {overview_out['patient_age_years']} years, "
    f"Gender: {overview_out['patient_gender']}) ==="
)
print(overview_out["answer"])

# Conditions
conditions_out = rag_new(
    query=conditions_question,
    patient_id=patient_id,
    question_type="conditions",
    num_results=5,
    model="gpt-5.4-mini",
    search_type="hybrid",
)

print(f"\n=== Conditions answer for patient {conditions_out['patient_name']} ===")
print(conditions_out["answer"])

# Medications
medications_out = rag_new(
    query=medications_question,
    patient_id=patient_id,
    question_type="medications",
    num_results=5,
    model="gpt-5.4-mini",
    search_type="hybrid",
)

print(f"\n=== Medications answer for patient {medications_out['patient_name']} ===")
print(medications_out["answer"])

# Oncology timeline (if configured)
oncology_out = rag_new(
    query=oncology_question,
    patient_id=patient_id,
    question_type="oncology_timeline",
    num_results=5,
    model="gpt-5.4-mini",
    search_type="hybrid",
)

print(f"\n=== Oncology timeline answer for patient {oncology_out['patient_name']} ===")
print(oncology_out["answer"])

=== Patient overview answer for Corrie32 Boyle917 (DOB 2020-12-18, Age: 5 years, Gender: female) ===
1. **Summary**
- The patient has documented breast cancer, specifically malignant neoplasm of breast with clinical stage 3A / stage 3 and HER2-negative disease, along with chemotherapy treatment history.
- The most recent documented status shows improvement in cancer disease progression and a treatment change noted on 2022-05-20.
- Oncology timeline: Breast cancer was documented as active on 2021-06-14 with stage 3A/stage 3 features, followed by repeated completed chemotherapy procedures from 2021-06-25 through 2022-05-14 and improved disease progression noted on 2021-12-06 and 2022-05-20.

2. **Conditions**
- **Malignant neoplasm of breast (disorder)** — active; date: 2021-06-14

3. **Medications**
- **Active oncology medications** — ribociclib, tamoxifen
- **Active chemotherapy agent** — epirubicin hydrochloride

4. **Oncology timeline**
- 2021-06-14 to 2021-06-15: Breast cancer docum

# aee216e6-cbe8-eaf2-3241-4bd1e8a01494

In [192]:
patient_id = "aee216e6-cbe8-eaf2-3241-4bd1e8a01494" #medium complexity patient

overview_question = "Provide a brief overview of this patient's medical background and current status."
conditions_question = "What are this patient's main diagnosed conditions and their status?"
medications_question = "What medications is this patient currently or recently taking?"
oncology_question = "Summarize this patient's oncology history."

In [193]:
# Overview 3rd version with new PATIENT_OVERVIEW_EXTRA

overview_out = rag_new(
    query=overview_question,
    patient_id=patient_id,
    question_type="patient_overview",
    num_results=5,
    model="gpt-5.4-mini",
    search_type="hybrid",
)

print(
    f"=== Patient overview answer for {overview_out['patient_name']} "
    f"(DOB {overview_out['patient_dob']}, Age: {overview_out['patient_age_years']} years, "
    f"Gender: {overview_out['patient_gender']}) ==="
)
print(overview_out["answer"])

# Conditions
conditions_out = rag_new(
    query=conditions_question,
    patient_id=patient_id,
    question_type="conditions",
    num_results=5,
    model="gpt-5.4-mini",
    search_type="hybrid",
)

print(f"\n=== Conditions answer for patient {conditions_out['patient_name']} ===")
print(conditions_out["answer"])

# Medications
medications_out = rag_new(
    query=medications_question,
    patient_id=patient_id,
    question_type="medications",
    num_results=5,
    model="gpt-5.4-mini",
    search_type="hybrid",
)

print(f"\n=== Medications answer for patient {medications_out['patient_name']} ===")
print(medications_out["answer"])

# Oncology timeline (if configured)
oncology_out = rag_new(
    query=oncology_question,
    patient_id=patient_id,
    question_type="oncology_timeline",
    num_results=5,
    model="gpt-5.4-mini",
    search_type="hybrid",
)

print(f"\n=== Oncology timeline answer for patient {oncology_out['patient_name']} ===")
print(oncology_out["answer"])

=== Patient overview answer for Deeann517 Torp761 (DOB 1984-06-18, Age: 42 years, Gender: female) ===
1. **Summary**
- The documented clinical conditions include an active breast malignancy, a resolved history of acute myeloid leukemia, resolved preeclampsia, and an active wrist sprain.  
- The most recent clinically important status is an active wrist sprain documented on 2022-05-31, with active use of analgesic medications in the medication snapshot from 2022-04-04 to 2022-05-31.  
- Oncology timeline documents breast cancer diagnosed as Stage 1A/HER2-positive with T1N0M0 features in 2013 and repeated radiotherapy course entries through 2013-12-17, with later observations repeatedly stating the condition improved.

2. **Conditions**
- **Sprain of wrist** — active; date: 2022-05-31
- **Preeclampsia** — resolved; date: 2017-05-01
- **Acute myeloid leukemia, disease (disorder)** — resolved; date: 1988-06-17
- **Malignant neoplasm of breast (disorder)** — active; date: 2013-10-11

3. **M

In [194]:
docs = _search(
    search_type="hybrid",
    query=overview_question,
    patient_id=patient_id,
    doc_types=["patient_overview"],
    num_results=5,
)

doc = docs[0]
print("TITLE:", doc.get("title"))
print("HEADING:", doc.get("heading"))
print("CHUNK TEXT:")
print(doc.get("chunk_text", ""))

TITLE: Patient Overview: Deeann517 Torp761
HEADING: Identity
CHUNK TEXT:
- Patient ID: aee216e6-cbe8-eaf2-3241-4bd1e8a01494
- Gender: female
- Birth date: 1984-06-18


In [195]:
print(_extract_patient_identity(docs))

('Deeann517 Torp761', '1984-06-18', 42, 'female')


In [196]:
# ...existing code...

question_type = "patient_overview"
query = overview_question

cfg = QUESTION_TYPES.get(question_type, {})
doc_types = cfg.get("doc_types") or cfg.get("doc_types_primary")
headings = cfg.get("headings") or cfg.get("headings_primary")

search_results = _search(
    search_type="hybrid",
    query=query,
    patient_id=patient_id,
    doc_types=doc_types,
    num_results=5,
)

# same fallback used in rag_new
if question_type == "patient_overview" and not _has_heading(search_results, "Medications"):
    medication_results = _search(
        search_type="hybrid",
        query="medications",
        patient_id=patient_id,
        doc_types=["patient_overview"],
        num_results=10,
    )
    medication_results = [d for d in medication_results if d.get("heading") == "Medications"]
    search_results = _dedupe_by_chunk_id(search_results + medication_results)

context = build_filtered_context(
    search_results=search_results,
    question_type=question_type,
    headings=headings,
)

print(context)

# optional: inspect what chunks formed the context
[(d.get("doc_type"), d.get("heading"), d.get("chunk_id")) for d in search_results]

patient_id: aee216e6-cbe8-eaf2-3241-4bd1e8a01494
doc_type: patient_overview
title: Patient Overview: Deeann517 Torp761
heading: Recent Conditions
date_start: 2017-03-06T18:00:41-05:00
date_end: 2022-05-31T18:08:50-04:00
is_oncology: 1
chunk_text: - Sprain of wrist; date: 2022-05-31T18:08:50-04:00; status: active
- Stress (finding); date: 2022-04-04T18:53:33-04:00; status: active
- Full-time employment (finding); date: 2022-04-04T18:53:33-04:00; status: active
- Part-time employment (finding); date: 2021-03-29T19:00:31-04:00; status: resolved
- Reports of violence in the environment (finding); date: 2020-03-23T18:55:34-04:00; status: resolved
- Full-time employment (finding); date: 2020-03-23T18:55:34-04:00; status: resolved
- Part-time employment (finding); date: 2019-03-18T18:54:21-04:00; status: resolved
- Not in labor force (finding); date: 2018-03-12T19:03:24-04:00; status: resolved
- Preeclampsia; date: 2017-05-01T18:08:50-04:00; status: resolved
- Part-time employment (finding); 

[('patient_overview', 'Identity', 'b0f73e949adfd7fe58626f90738c565ab2da47df'),
 ('patient_overview',
  'Recent Encounters',
  '81ad73641c81ec6c67052ae1c54da5a9b68b50a3'),
 ('patient_overview',
  'Diagnostic Reports',
  '63658fec47ac14532433f60320a173fd6fa6d33d'),
 ('patient_overview',
  'Provenance',
  '1a0a82a1caaaae3029bb150440e943f921e825fd'),
 ('patient_overview',
  'Recent Conditions',
  'e733276d7db1b542f556b25839e3602be1339cbc'),
 ('patient_overview',
  'Medications',
  '3b6a76df9116342cca819342b5a77a629c04d636')]

# FOR THE APP

# BOLD for 'active'

# links

## Patient name - KEEP AS IS
You filtered Identity out of the overview’s LLM context, which is correct for clinical summarization, but your display layer should obtain demographics directly from patient.csv or a dedicated identity lookup—not from the filtered RAG results.

# FIXES FOR QUESTION_TYPES

Compacted conversationI’m tracing the routing and prompt construction around the medication path so I can pinpoint why the model is still emitting the fallback text.

Searched for regex `def build_context|def llm|def search\(|def hybrid_search\(|INSTRUCTIONS|No medication information documented|patient_overview` (`**/clinical_synopsis/**/*.py`), 42 results

Read [](file:///Users/barbarato/llm-zoomcamp-capstone-project/clinical_synopsis/rag.py#1-1), lines 1 to 400

I’m verifying the actual retrieved documents for a sample patient to see whether the medication section is being filtered out before the prompt is even sent.

Got last terminal command

Checked [](file:///Users/barbarato/llm-zoomcamp-capstone-project/notebooks/rag_eval_routing_questions.ipynb), 5 problems found

Read terminal selection

Got last terminal command

Got last terminal command

Searched for regex `def rag\(|rag\s*=\s*` (`**/*.{py,ipynb}`), 5 results

Searched for regex `^## Medications|^## Medication|Medications` (`**/data/**`), 2 results

Searched for regex `"heading": "Medications"|"heading":"Medications"|"doc_type": "patient_overview"` (`**/data/retrieval/*`), no results

Read [](file:///Users/barbarato/llm-zoomcamp-capstone-project/clinical_synopsis/rag.py#400-400), lines 400 to 509

Read [](file:///Users/barbarato/llm-zoomcamp-capstone-project/data/derived/sample50/f203e11d-5573-1624-69b8-af8436987b3e/patient_overview.md)

Searched for text `Medications` (`**/data/retrieval/**`), 50 results

Read [](file:///Users/barbarato/llm-zoomcamp-capstone-project/clinical_synopsis/build_retrieval_metadata_db.py#1-1), lines 1 to 220

The pattern is becoming clear: the filter is matching headings too literally, and the retrieved data appears to be indexed with a different case from the notebook’s configuration.

Read terminal selection

Got last terminal command

The issue is not the wording of the prompt itself; it is that the medication section is not actually being used in the retrieval/filtering path for the overview flow.

What is happening:
- In rag_eval_routing_questions.ipynb, your `rag_new` function reads:
  - `cfg.get("doc_types", None)`
  - `cfg.get("headings", None)`
- But for the `patient_overview` config, the values are stored under:
  - `doc_types_primary`
  - `headings_primary`

So for that archetype, `doc_types` becomes `None` and `headings` becomes `None`. That means:
- retrieval is not constrained to the intended patient-overview docs,
- the heading filter is not applied,
- and with `num_results=5`, the medication chunk may not be included in the context.

Once that happens, the model follows your own fallback rule and emits:
- “No medication information documented.”

The fix is to read the right config keys:

```python
cfg = QUESTION_TYPES.get(question_type, {})
prompt_mode = cfg.get("prompt_mode", "summary")
doc_types = cfg.get("doc_types") or cfg.get("doc_types_primary")
headings = cfg.get("headings") or cfg.get("headings_primary")
```

If you want, I can also help you make the filter more robust by normalizing heading case, so values like `Medications` and `medications` both match.

# FIXES for rag_new

In rag_eval_routing_questions.ipynb:1179, you define doc_types_med_fallback and doc_types_onco_fallback, but the active rag_new cell in rag_eval_routing_questions.ipynb:1606 never reads or uses them. So the overview path still does one retrieval pass, filters that result set once, and never merges in medications or oncology_timeline as a second source when patient_overview is sparse.

I’m narrowing this to the first routing gap only: the notebook does not guarantee the Medications section will be in the retrieved top-N, so the later prompt can never see it.

The first problem is that `rag_new` is only doing a single retrieval pass, and it is using that pass as the only source of truth for the prompt.

In your notebook, the flow is:

1. `rag_new(...)` calls `rag_module.hybrid_search(...)` with the question text.
2. That returns only the top `num_results=5` chunks for the patient-overview document.
3. `build_filtered_context(...)` then keeps only chunks whose heading matches the requested headings.
4. The prompt is built from whatever survived step 3.

That means `Medications` is not fetched by section name. It is only included if a Medications chunk happened to rank high enough in the initial top 5 results. If the overview question pulls in Recent Conditions, Recent Results, or Recent Encounters first, the Medications chunk can simply fall below the cutoff and never enter the context at all.

So the key point is: headings are a filter on already-retrieved chunks, not a retrieval guarantee. The notebook is not asking for the Medications section directly, and it is not doing a second pass when Medications is missing. It is relying on the first retrieval to surface it.

Why that matters in practice:
- patient_overview.md does contain a Medications section.
- But `rag_new` does not read the whole document.
- It only sees the few retrieved chunks returned by `hybrid_search`.
- If those retrieved chunks do not include the Medications block, the model gets no medication text, even though the source document has it.

So the failure is not “Medications is absent from patient_overview.” It is “Medications is present, but the retrieval step is not guaranteed to include that section before filtering.”

If you want, I can point to the exact notebook cell logic that causes this behavior and describe the minimal notebook-only change that would make Medications reliably appear.

Replace the active rag_new cell in rag_eval_routing_questions.ipynb:1606 with this version in (2) rag_new. It keeps your current flow, but adds a second, section-targeted retrieval pass only when the first pass does not surface Medications.

# Note on Patient_Overview.md

**“Recent” = top 10 diagnostic reports.**



Thanks for pasting the script—that makes it clear how `patient_overview.md` and the oncology timeline were built.

## How patient_overview.md is generated

For each patient folder under `data/interim/sample50/{patient_id}`:

1. **Load normalized CSVs derived from the FHIR Bundle**

   ```python
   patient_df        = safe_read_csv(patient_dir / "patient.csv")
   encounters_df     = safe_read_csv(patient_dir / "encounters.csv")
   conditions_df     = safe_read_csv(patient_dir / "conditions.csv")
   observations_df   = safe_read_csv(patient_dir / "observations.csv")
   med_req_df        = safe_read_csv(patient_dir / "medication_requests.csv")
   med_admin_df      = safe_read_csv(patient_dir / "medication_administrations.csv")
   procedures_df     = safe_read_csv(patient_dir / "procedures.csv")
   reports_df        = safe_read_csv(patient_dir / "diagnostic_reports.csv")
   metadata          = json.loads((patient_dir / "bundle_metadata.json").read_text(...))
   ```

   These CSVs are themselves derived from the original FHIR Bundle (Condition, Observation, MedicationRequest, MedicationAdministration, Procedure, DiagnosticReport, Encounter, Patient, etc.).

2. **Combine medication requests and administrations**

   ```python
   meds_df = combine_medications(med_req_df, med_admin_df)
   ```

   `combine_medications`:

   - Tags rows as `MedicationRequest` or `MedicationAdministration`.
   - Adds `event_date` based on authored_on / effective_datetime / effective_period_start.
   - Sorts descending by event_date.

3. **Summarize each clinical table into bullet lists**

   - Conditions → `summarize_conditions(conditions_df)`
   - Observations (labs/results) → `summarize_observations(observations_df)`
   - Encounters → `summarize_encounters(encounters_df)`
   - Medications → `summarize_meds(meds_df)`
   - Procedures → `summarize_procedures(procedures_df)`
   - Diagnostic reports → `summarize_reports(reports_df)`

   Each `summarize_*` function:

   - Sorts by a relevant date column (`sort_by_date`).
   - Picks a display label from `display`, `text`, or `code`.
   - Adds date and status/value when available.
   - Emits Markdown bullets like:

     - Conditions: `- {label}; date: {when}; status: {clinical_status/verification_status}`
     - Meds: `- {label}; date: {when}; status: {status/intent}`
     - Observations: `- {label}; value: ...; date: {when}`
     - Encounters: `- {encounter_type_display}; start: ...; end: ...; provider: ...`
     - Procedures / Reports: `- {label}; date: {when}`

4. **Build the patient overview document**

   `build_overview_md(...)` constructs the Markdown:

   - Identity:

     ```markdown
     # Patient Overview: {patient_display_name}
     
     ## Identity
     - Patient ID: {patient_id}
     - Gender: ...
     - Birth date: ...
     ```

   - Record snapshot (counts by resource type):

     ```markdown
     ## Record Snapshot
     - Source bundle: {source_file}
     - Bundle entries: {n_entries}
     - Encounters: {len(encounters_df)}
     - Conditions: {len(conditions_df)}
     - Observations: {len(observations_df)}
     - Medications: {len(meds_df)}
     - Procedures: {len(procedures_df)}
     - Diagnostic reports: {len(reports_df)}
     ```

   - Clinical sections:

     ```markdown
     ## Recent Conditions
     {summarize_conditions bullets or "- No condition records found."}

     ## Recent Results
     {summarize_observations bullets or "- No observation records found."}

     ## Recent Encounters
     {summarize_encounters bullets or "- No encounter records found."}

     ## Medications
     {summarize_meds bullets}  # only if meds_df has rows

     ## Procedures
     {summarize_procedures bullets}  # only if procedures_df has rows

     ## Diagnostic Reports
     {summarize_reports bullets}  # only if reports_df has rows
     ```

   - Provenance:

     ```markdown
     ## Provenance
     - This document is derived from normalized CSV tables in the same patient folder.
     - Use the `source_file` and `resource_id` columns in CSV files to trace facts back to the original FHIR bundle.
     ```

   Finally, it writes:

   ```python
   (out_dir / "patient_overview.md").write_text(overview_md, encoding="utf-8")
   ```

So `patient_overview.md` is a **rule-based patient summary** directly generated from FHIR-derived CSVs, not LLM-written text. It compresses the long-term records into:

- Identity + snapshot
- Recent Conditions
- Recent Results
- Recent Encounters
- Medications
- Procedures
- Diagnostic Reports
- Provenance

## How the oncology timeline is generated

You also built an oncology-specific timeline:

1. **Collect oncology-related events from all tables**

   ```python
   oncology_events_df = collect_oncology_events(
       conditions_df=conditions_df,
       observations_df=observations_df,
       meds_df=meds_df,
       procedures_df=procedures_df,
       reports_df=reports_df,
   )
   ```

   `collect_oncology_events`:

   - Uses keyword matching (`text_matches_oncology`) over labels/status/value to decide if a row is oncology-related.
   - For each oncology hit, creates an event with:
     - `event_type` (Condition / Observation / Medication / Procedure / DiagnosticReport),
     - `date`,
     - `label`,
     - `status` (clinical_status, value, or status/intent),
     - `resource_id` and `source_file` for provenance.
   - Sorts events chronologically (ascending by date).

2. **Build the oncology timeline Markdown and events CSV**

   If there are at least 3 dated events:

   ```python
   timeline_md = build_oncology_timeline_md(patient_id, patient_name, oncology_events_dated)
   (out_dir / "oncology_timeline.md").write_text(timeline_md, encoding="utf-8")
   oncology_events_dated.to_csv(out_dir / "oncology_timeline_events.csv", index=False)
   ```

   `build_oncology_timeline_md`:

   - Header + summary:
     ```markdown
     # Oncology Timeline: {patient_name}
     
     - Patient ID: {patient_id}
     - Oncology-related dated events: {len(events_df)}
     ```
   - Timeline:
     ```markdown
     ## Timeline
     - {date} — {event_type}: {label}; detail: {status}; resource_id: {resource_id}
     ```
   - Provenance:
     ```markdown
     ## Provenance
     - Each timeline event was selected from normalized source tables by keyword matching.
     - Confirm clinical relevance against the original CSV rows before downstream use.
     ```

This is the `oncology_timeline.md` and `oncology_timeline_events.csv` you’re now using as oncology-specific context.

## How this connects to what you’re seeing now

- `patient_overview.md` is a **lossy but structured view** of the long-term CSV records:
  - It truncates to “recent” entries using date sorting and head limits (e.g., top 10 conditions, top 12 observations, top 8 encounters, etc.).
  - It doesn’t show *all* historical records; just the most recent slice.
- The full CSVs still contain additional long-term conditions/medications/observations that **do not appear in patient_overview**, which explains why you see gaps when you compare RAG answers (based only on patient_overview) to the full CSV history.
- For your current RAG design, you’re rightly treating `patient_overview.md` as the **trusted summary source** and then offering CSVs as “click for more detail,” rather than trying to have the LLM synthesize everything from raw tables every time.

If you want, we can now:

- Adjust your `CONDITIONS_EXTRA` / `MEDICATIONS_EXTRA` guidance to explicitly mention that patient_overview is a *recent snapshot* and that long-term history may require consulting the CSVs.
- Or add an additional archetype (e.g., “long_term_history”) that uses CSV-backed doc_types instead of just patient_overview, so you can ask the model questions specifically about full history versus recent snapshot.


“Recent” in `patient_overview.md` is defined **purely by sorting by date and then truncating to a fixed number of rows per section**. There is **no calendar cutoff** (e.g., “last 1 year”); it’s “most recent N entries” per resource type.

Here is the exact behavior for each section:

## Conditions (Recent Conditions)

```python
def summarize_conditions(df: pd.DataFrame, limit: int = 10) -> list[str]:
    if not has_rows(df):
        return []
    df2 = sort_by_date(df, ["recorded_date", "onset_datetime"])
    lines = []
    for _, row in df2.head(limit).iterrows():
        ...
```

- Sort order: descending by `recorded_date`, then `onset_datetime` (first column that exists).
- “Recent” = **top 10 rows** after that sort.
- Anything older than the 10 most recent conditions is **not shown** in `Recent Conditions`.

## Observations (Recent Results)

```python
def summarize_observations(df: pd.DataFrame, limit: int = 12) -> list[str]:
    if not has_rows(df):
        return []
    df2 = sort_by_date(df, ["effective_datetime", "issued"])
    lines = []
    for _, row in df2.head(limit).iterrows():
        ...
```

- Sort order: descending by `effective_datetime`, then `issued`.
- “Recent” = **top 12 rows**.
- Older observations are omitted from `Recent Results`.

## Encounters (Recent Encounters)

```python
def summarize_encounters(df: pd.DataFrame, limit: int = 8) -> list[str]:
    if not has_rows(df):
        return []
    df2 = sort_by_date(df, ["period_start", "period_end"])
    lines = []
    for _, row in df2.head(limit).iterrows():
        ...
```

- Sort order: descending by `period_start`, then `period_end`.
- “Recent” = **top 8 encounters**.

## Medications (Medications section)

```python
def summarize_meds(df: pd.DataFrame, limit: int = 10) -> list[str]:
    if not has_rows(df):
        return []
    df2 = sort_by_date(df, ["authored_on", "effective_datetime", "effective_period_start"])
    lines = []
    for _, row in df2.head(limit).iterrows():
        ...
```

- Sort order: descending by `authored_on`, then `effective_datetime`, then `effective_period_start`.
- “Recent” (for meds) = **top 10 medication events** from the combined MedicationRequest/MedicationAdministration table.

## Procedures (Procedures section)

```python
def summarize_procedures(df: pd.DataFrame, limit: int = 10) -> list[str]:
    if not has_rows(df):
        return []
    df2 = sort_by_date(df, ["performed_datetime", "performed_period_start"])
    lines = []
    for _, row in df2.head(limit).iterrows():
        ...
```

- Sort order: descending by `performed_datetime`, then `performed_period_start`.
- “Recent” = **top 10 procedures**.

## Diagnostic Reports (Diagnostic Reports section)

```python
def summarize_reports(df: pd.DataFrame, limit: int = 10) -> list[str]:
    if not has_rows(df):
        return []
    df2 = sort_by_date(df, ["effective_datetime", "issued"])
    lines = []
    for _, row in df2.head(limit).iterrows():
        ...
```

- Sort order: descending by `effective_datetime`, then `issued`.
- “Recent” = **top 10 diagnostic reports**.

## Takeaway

- “Recent” = **most recent N entries by date**, where:
  - N = 10 for conditions, meds, procedures, reports.
  - N = 12 for observations.
  - N = 8 for encounters.
- There is **no explicit time window** (like “last year”); it is strictly a **top-N cutoff based on the available dates**.
- That’s why older long-term history in the CSVs may be missing from `patient_overview.md`: it only shows the latest slice per category.